# Chat.DT on IFC-Bench — everything in this Colab VM

Neo4j, the graph snapshots, the runner and (for the local-model arm) the GPU model all live in this one runtime. No server, no tunnel, no bundles.

Per model: restore its graph snapshot (built here by the ETL the first time) → build the prompt context from the live graph → run every question × paradigm → judge with the IFC-Bench judge.

**Colab Secrets** (key icon): `OPENROUTER_API_KEY` (required); `GITHUB_TOKEN` only if the repo is private.

Snapshots and results are kept on Google Drive, so a dropped session resumes where it stopped: re-run the cells, finished cells are skipped.

In [ ]:
# 1. Code + secrets
import os, subprocess
from google.colab import userdata

BRANCH = 'journal/agentic-cfg'
REPO = 'github.com/IvansSmirnoff/Chat.DT.git'

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
if not os.path.isdir('/content/Chat.DT'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, '/content/Chat.DT'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Chat.DT', 'pull', '-q', '--ff-only'], check=True)
os.chdir('/content/Chat.DT')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

os.environ['OPENROUTER_API_KEY'] = secret('OPENROUTER_API_KEY') or ''
assert os.environ['OPENROUTER_API_KEY'], 'Add OPENROUTER_API_KEY to Colab Secrets'
os.environ.update({
    'NEO4J_URI': 'bolt://127.0.0.1:7687', 'NEO4J_USER': 'neo4j', 'NEO4J_PASSWORD': 'password',
    'IFCBENCH_MANIFEST_PATH': '/content/Chat.DT/data/ifcbench/manifest.yaml',
})

In [ ]:
# 2. Dependencies + Neo4j (~2 min on a fresh VM; a re-run only restarts Neo4j)
!pip install -q -r requirements-base.txt
# bootstrap.sh sizes Neo4j to the VM's RAM (NEO4J_HEAP_MAX / NEO4J_PAGECACHE override it).
!bash scripts/colab/bootstrap.sh

In [ ]:
# 3. Persist snapshots + results on Drive (resume across sessions)
from google.colab import drive
drive.mount('/content/drive')
PERSIST = '/content/drive/MyDrive/chatdt'
for local, remote in [('data/ifcbench/cache', 'ifcbench/cache'),
                      ('results/ifcbench/runs', 'ifcbench/runs')]:
    os.makedirs(f'{PERSIST}/{remote}', exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.islink(local):
        subprocess.run(['rm', '-rf', local])
        os.symlink(f'{PERSIST}/{remote}', local)
    print(local, '->', os.readlink(local))

In [ ]:
# 4. IFC files from HuggingFace (pinned dataset commit, sha256-verified).
#    --project dental_clinic for Tier A; --split test for all 31 test-split files (1.34 GB).
!python scripts/ifcbench/fetch.py --project dental_clinic

In [ ]:
# 5. Run. Missing snapshots are built here (ETL + export) the first time, then restored.
#    Swap --model-id for --manifest-split test to sweep every model.
LLM = 'openrouter:z-ai/glm-4.5-air'   # Hellin backbones: z-ai/glm-5.2, minimax/minimax-m3, z-ai/glm-4.5-air
PROVIDER = 'z-ai'                      # pin the OpenRouter provider; unpinned calls drift between providers
REASONING = 'off'                      # run factor: on | off (same in both arms)
RUN = 'clinic_glm45air_' + ('think' if REASONING == 'on' else 'nothink')
!python -m src.agent.run --model-id dental_clinic__arc --model-id dental_clinic__str \
    --split test --llm {LLM} --provider {PROVIDER} --reasoning {REASONING} \
    --judge-provider z-ai --run {RUN} --workers 6

In [ ]:
# 6. Inspect: per-question verdicts and the judge's reasons
import json, pandas as pd
rows = [json.loads(l) for l in open(f'results/ifcbench/runs/{RUN}/episodes.jsonl')]
df = pd.DataFrame([{
    'model': r['model_id'], 'q': r['question_id'], 'cat': r['category'], 'paradigm': r['paradigm'],
    'verdict': (r['judgement'] or {}).get('verdict'), 'queries': r['episode']['n_queries'],
    'cost': round(r['episode']['cost_usd'], 4), 'question': r['question'][:70],
    'why': (r['judgement'] or {}).get('justification', '')[:120],
} for r in rows])
display(df.pivot_table(index='paradigm', columns='verdict', values='q', aggfunc='count', fill_value=0))
df.sort_values(['q', 'paradigm'])